# Build CatBoost customer-product ranking data

## Goal

Create one ranking group for each customer purchase date. Every paid product on that date is retained with `label = 1`, including first purchases. Gift rows have already been removed. A small, reproducible set of real purchasable products receives `label = 0`:

- products the same customer bought earlier but did not buy on the current date;
- catalogue products the customer had never paid for before.

Every historical value is calculated from the same customer-product pair strictly before the group date. No customer, product history, or date is substituted. Dates are kept only as group metadata and are not model features. All numeric missing values are stored as zero.

## 1. Setup

In [38]:
from pathlib import Path
import random

import numpy as np
import pandas as pd

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
input_path = project_root / "data" / "interim" / "cleaned_purchases.csv"
training_output_path = project_root / "data" / "processed" / "catboost_training.csv"
group_metadata_path = project_root / "data" / "processed" / "catboost_group_metadata.csv"

random_seed = 42
target_candidates_per_group = 25
historical_negative_fraction = 0.5
product_recent_window_days = 30

## 2. Load, validate, and build the product catalogue

Source rows are sorted before history is built. The catalogue contains only real products and records when each product first appeared, so negative sampling cannot select a product before it existed in the data.

In [39]:
identifier_dtypes = {
    "customer_id": "string",
    "product_id": "string",
    "product_category": "string",
    "business_line": "string",
}

purchases = pd.read_csv(
    input_path,
    dtype=identifier_dtypes,
    parse_dates=["purchase_date"],
)

required_columns = {
    "customer_id",
    "purchase_date",
    "product_id",
    "quantity",
    "product_category",
    "business_line",
}
missing_columns = sorted(required_columns - set(purchases.columns))
if missing_columns:
    raise ValueError(f"Missing expected columns: {missing_columns}")

event_key_columns = ["customer_id", "purchase_date", "product_id"]
purchases = (
    purchases
    .sort_values(event_key_columns)
    .reset_index(drop=True)
)

duplicate_count = int(purchases.duplicated(event_key_columns).sum())
non_positive_purchase_count = int(purchases["quantity"].le(0).sum())
assert duplicate_count == 0, f"Found {duplicate_count} duplicate event keys."
assert non_positive_purchase_count == 0, (
    f"Found {non_positive_purchase_count} non-positive purchases."
)

product_catalogue = (
    purchases
    .sort_values(["purchase_date", "product_id"])
    .groupby("product_id", sort=False)
    .agg(
        product_category=("product_category", "first"),
        business_line=("business_line", "first"),
        first_seen_date=("purchase_date", "min"),
    )
)
catalogue_products = sorted(product_catalogue.index.tolist())
product_first_seen = product_catalogue["first_seen_date"].to_dict()

product_daily_history = {}
for product_id, product_events in purchases.groupby("product_id", sort=False):
    daily = (
        product_events
        .groupby("purchase_date", sort=True)
        .agg(purchase_count=("customer_id", "size"))
        .reset_index()
    )
    product_daily_history[product_id] = {
        "dates": daily["purchase_date"].to_numpy(dtype="datetime64[ns]"),
        "purchase_count_cumulative": daily["purchase_count"].cumsum().to_numpy(),
    }

product_customer_first_purchase_dates = {
    product_id: np.sort(product_events.to_numpy(dtype="datetime64[ns]"))
    for product_id, product_events in (
        purchases
        .groupby(["product_id", "customer_id"])["purchase_date"]
        .min()
        .groupby(level="product_id")
    )
}

print(f"Loaded {len(purchases):,} sorted paid-purchase rows.")
print(f"Built a catalogue of {len(product_catalogue):,} real products.")

Loaded 71,916 sorted paid-purchase rows.
Built a catalogue of 927 real products.


## 3. Create customer-date groups with truthful history

Each group belongs to exactly one customer and one purchase date. Positive candidates are the products actually paid for on that date. The sampler targets 25 candidates, balances historical and never-paid negatives when possible, and retains every positive.

All customer and product-history features use only events strictly before the scoring date. Added signals cover robust reorder cadence, category and business-line depth, customer history length, and recent product demand.

In [40]:
def new_product_history():
    return {
        "purchase_count": 0,
        "quantity_sum": 0.0,
        "last_quantity": 0.0,
        "last_purchase_date": None,
        "interval_days_sum": 0,
        "interval_count": 0,
        "interval_days": [],
        "std_interval_days": 0.0,
    }


random_generator = random.Random(random_seed)
candidate_rows = []
group_metadata_rows = []
next_group_id = 0

for customer_id, customer_events in purchases.groupby("customer_id", sort=False):
    product_history = {}
    customer_paid_products = set()
    category_purchase_counts = {}
    business_line_purchase_counts = {}
    previous_customer_item_purchase_count = 0

    for scoring_date, date_events in customer_events.groupby(
        "purchase_date",
        sort=False,
    ):
        paid_products = set(
            date_events.loc[
                date_events["quantity"].gt(0),
                "product_id",
            ]
        )

        if paid_products:
            negative_target = max(
                target_candidates_per_group - len(paid_products),
                1,
            )
            historical_pool = sorted(customer_paid_products - paid_products)
            historical_sample_size = min(
                round(negative_target * historical_negative_fraction),
                len(historical_pool),
            )
            historical_negatives = set(
                random_generator.sample(
                    historical_pool,
                    historical_sample_size,
                )
            )

            available_products = {
                product_id
                for product_id in catalogue_products
                if product_first_seen[product_id] <= scoring_date
            }
            never_paid_pool = sorted(
                available_products
                - customer_paid_products
                - paid_products
            )
            never_paid_sample_size = min(
                negative_target - historical_sample_size,
                len(never_paid_pool),
            )
            never_paid_negatives = set(
                random_generator.sample(
                    never_paid_pool,
                    never_paid_sample_size,
                )
            )
            remaining_negative_slots = (
                negative_target
                - len(historical_negatives)
                - len(never_paid_negatives)
            )
            if remaining_negative_slots > 0:
                historical_backfill_pool = sorted(
                    set(historical_pool) - historical_negatives
                )
                historical_negatives.update(
                    random_generator.sample(
                        historical_backfill_pool,
                        min(remaining_negative_slots, len(historical_backfill_pool)),
                    )
                )

            group_products = sorted(
                paid_products
                | historical_negatives
                | never_paid_negatives
            )

            for product_id in group_products:
                history = product_history.get(product_id)
                category = product_catalogue.at[
                    product_id,
                    "product_category",
                ]
                business_line = product_catalogue.at[
                    product_id,
                    "business_line",
                ]

                if history is None:
                    previous_paid_purchase_count = 0
                    previous_quantity_sum = 0.0
                    last_paid_quantity = 0.0
                    days_since_last_paid_purchase = 0
                    average_days_between_customer_product_purchases = 0.0
                    std_days_between_customer_product_purchases = 0.0
                    observed_reorder_interval_count = 0
                    has_replenishment_cycle = False
                    expected_days_before_next_order = 0.0
                else:
                    previous_paid_purchase_count = history[
                        "purchase_count"
                    ]
                    previous_quantity_sum = history["quantity_sum"]
                    last_paid_quantity = history["last_quantity"]
                    last_paid_purchase_date = history["last_purchase_date"]
                    days_since_last_paid_purchase = (
                        scoring_date - last_paid_purchase_date
                    ).days
                    average_days_between_customer_product_purchases = (
                        history["interval_days_sum"]
                        / history["interval_count"]
                        if history["interval_count"] > 0
                        else 0.0
                    )
                    std_days_between_customer_product_purchases = history[
                        "std_interval_days"
                    ]
                    observed_reorder_interval_count = history["interval_count"]
                    average_paid_quantity = (
                        previous_quantity_sum
                        / previous_paid_purchase_count
                    )
                    has_replenishment_cycle = (
                        average_days_between_customer_product_purchases > 0
                        and last_paid_quantity > 0
                        and average_paid_quantity > 0
                    )
                    expected_days_before_next_order = (
                        average_days_between_customer_product_purchases
                        * last_paid_quantity
                        / average_paid_quantity 
                        - days_since_last_paid_purchase
                        if has_replenishment_cycle
                        else 0.0
                    )

                if product_id in paid_products:
                    candidate_type = "positive"
                elif product_id in historical_negatives:
                    candidate_type = "historical_negative"
                else:
                    candidate_type = "never_paid_negative"

                candidate_rows.append({
                    "group_id": next_group_id,
                    "customer_id": customer_id,
                    "product_id": product_id,
                    "product_category": category,
                    "business_line": business_line,
                    "previous_paid_purchase_count": (
                        previous_paid_purchase_count
                    ),
                    "previous_paid_quantity": previous_quantity_sum,
                    "last_paid_quantity": last_paid_quantity,
                    "days_since_last_paid_purchase": (
                        days_since_last_paid_purchase
                    ),
                    "average_days_between_customer_product_purchases": (
                        average_days_between_customer_product_purchases
                    ),
                    "std_days_between_customer_product_purchases": (
                        std_days_between_customer_product_purchases
                    ),
                    "observed_reorder_interval_count": (
                        observed_reorder_interval_count
                    ),
                    "expected_days_before_next_order": expected_days_before_next_order,
                    "previous_category_purchase_count": (
                        category_purchase_counts.get(category, 0)
                        if pd.notna(category)
                        else 0
                    ),
                    "previous_category_purchase_share": (
                        category_purchase_counts.get(category, 0)
                        / previous_customer_item_purchase_count
                        if pd.notna(category)
                        and previous_customer_item_purchase_count > 0
                        else 0.0
                    ),
                    "previous_business_line_purchase_count": (
                        business_line_purchase_counts.get(business_line, 0)
                        if pd.notna(business_line)
                        else 0
                    ),
                    "previous_business_line_purchase_share": (
                        business_line_purchase_counts.get(business_line, 0)
                        / previous_customer_item_purchase_count
                        if pd.notna(business_line)
                        and previous_customer_item_purchase_count > 0
                        else 0.0
                    ),
                    "label": int(product_id in paid_products),
                    "_candidate_type": candidate_type,
                    "_had_previous_paid_purchase": (
                        previous_paid_purchase_count > 0
                    ),
                    "_has_replenishment_cycle": has_replenishment_cycle,
                    "_scoring_date": scoring_date,
                })

            group_metadata_rows.append({
                "group_id": next_group_id,
                "customer_id": customer_id,
                "scoring_date": scoring_date,
            })
            next_group_id += 1

        # Current-date events become history only after candidates are built.
        for event in date_events.itertuples(index=False):
            history = product_history.setdefault(
                event.product_id,
                new_product_history(),
            )

            previous_purchase_date = history["last_purchase_date"]
            if previous_purchase_date is not None:
                interval_days = (scoring_date - previous_purchase_date).days
                history["interval_days_sum"] += interval_days
                history["interval_count"] += 1
                history["interval_days"].append(interval_days)
                history["std_interval_days"] = float(
                    np.std(history["interval_days"], ddof=0)
                )
            history["purchase_count"] += 1
            history["quantity_sum"] += event.quantity
            history["last_quantity"] = event.quantity
            history["last_purchase_date"] = scoring_date
            customer_paid_products.add(event.product_id)
            if pd.notna(event.product_category):
                category_purchase_counts[event.product_category] = (
                    category_purchase_counts.get(event.product_category, 0) + 1
                )
            if pd.notna(event.business_line):
                business_line_purchase_counts[event.business_line] = (
                    business_line_purchase_counts.get(event.business_line, 0) + 1
                )
            previous_customer_item_purchase_count += 1

candidate_data = pd.DataFrame(candidate_rows)
group_metadata = pd.DataFrame(group_metadata_rows)

if candidate_data.empty:
    raise ValueError("No candidate rows were created.")


def cumulative_before(cumulative_values, index):
    return cumulative_values[index - 1] if index > 0 else 0


for product_id, row_indexes in candidate_data.groupby("product_id").groups.items():
    history = product_daily_history[product_id]
    dates = history["dates"]
    query_dates = candidate_data.loc[row_indexes, "_scoring_date"].to_numpy(
        dtype="datetime64[ns]"
    )
    prior_indexes = np.searchsorted(dates, query_dates, side="left")
    recent_start_dates = query_dates - np.timedelta64(product_recent_window_days, "D")
    recent_start_indexes = np.searchsorted(dates, recent_start_dates, side="left")

    purchase_cumulative = history["purchase_count_cumulative"]
    lifetime_purchase_counts = np.array([
        cumulative_before(purchase_cumulative, index)
        for index in prior_indexes
    ])
    purchase_counts_before_recent = np.array([
        cumulative_before(purchase_cumulative, index)
        for index in recent_start_indexes
    ])
    recent_purchase_counts = lifetime_purchase_counts - purchase_counts_before_recent

    first_purchase_dates = product_customer_first_purchase_dates[product_id]
    unique_customer_counts = np.searchsorted(
        first_purchase_dates,
        query_dates,
        side="left",
    )
    candidate_data.loc[row_indexes, "historical_product_purchase_count"] = lifetime_purchase_counts
    candidate_data.loc[row_indexes, "historical_product_unique_customer_count"] = unique_customer_counts
    candidate_data.loc[row_indexes, "product_purchase_count_last_30_days"] = recent_purchase_counts

## 4. Validate and save

The checks prove that every source purchase survives as a positive, historical negatives have genuine prior paid history, never-paid negatives have zero prior paid history, and all groups contain both labels. Diagnostic candidate types are removed before saving.

In [41]:
positive_mask = candidate_data["_candidate_type"].eq("positive")
historical_negative_mask = candidate_data["_candidate_type"].eq(
    "historical_negative"
)
never_paid_negative_mask = candidate_data["_candidate_type"].eq(
    "never_paid_negative"
)

expected_positive_rows = len(purchases)
expected_groups = int(
    purchases[["customer_id", "purchase_date"]]
    .drop_duplicates()
    .shape[0]
)

assert int(candidate_data["label"].sum()) == expected_positive_rows
assert int(positive_mask.sum()) == expected_positive_rows
assert candidate_data.loc[positive_mask, "label"].eq(1).all()
assert candidate_data.loc[
    historical_negative_mask,
    "label",
].eq(0).all()
assert candidate_data.loc[
    historical_negative_mask,
    "_had_previous_paid_purchase",
].all()
assert candidate_data.loc[
    never_paid_negative_mask,
    "label",
].eq(0).all()
assert candidate_data.loc[
    never_paid_negative_mask,
    "_had_previous_paid_purchase",
].eq(False).all()

training_columns = [
    "group_id",
    "customer_id",
    "product_id",
    "product_category",
    "business_line",
    "previous_paid_purchase_count",
    "previous_paid_quantity",
    "last_paid_quantity",
    "days_since_last_paid_purchase",
    "average_days_between_customer_product_purchases",
    "std_days_between_customer_product_purchases",
    "observed_reorder_interval_count",
    "expected_days_before_next_order",
    "previous_category_purchase_count",
    "previous_category_purchase_share",
    "previous_business_line_purchase_count",
    "previous_business_line_purchase_share",
    "historical_product_purchase_count",
    "historical_product_unique_customer_count",
    "product_purchase_count_last_30_days",
    "label",
]
training_data = (
    candidate_data[training_columns]
    .sort_values(["group_id", "product_id"])
    .reset_index(drop=True)
)

forbidden_training_columns = {
    "purchase_date",
    "scoring_date",
    "quantity",
    "paid_quantity",
    "received_quantity",
    "gift_quantity",
    "bonus_amount",
    "promotion_name",
    "month",
    "weekday",
    "_candidate_type",
    "_had_previous_paid_purchase",
    "_has_replenishment_cycle",
    "_scoring_date",
}
assert forbidden_training_columns.isdisjoint(training_data.columns)
assert training_data["group_id"].is_monotonic_increasing
assert training_data.groupby("group_id")["customer_id"].nunique().eq(1).all()
numeric_columns = training_data.select_dtypes(include="number").columns
training_data[numeric_columns] = training_data[numeric_columns].fillna(0)
assert training_data[numeric_columns].notna().all().all()
assert training_data["previous_paid_purchase_count"].ge(0).all()
assert training_data["previous_paid_quantity"].ge(0).all()
assert training_data["last_paid_quantity"].ge(0).all()
assert training_data["days_since_last_paid_purchase"].ge(0).all()
assert training_data[
    "average_days_between_customer_product_purchases"
].ge(0).all()
assert training_data["std_days_between_customer_product_purchases"].ge(0).all()
assert training_data["observed_reorder_interval_count"].ge(0).all()
assert training_data["previous_category_purchase_count"].ge(0).all()
assert training_data["previous_category_purchase_share"].between(0, 1).all()
assert training_data["previous_business_line_purchase_count"].ge(0).all()
assert training_data["previous_business_line_purchase_share"].between(0, 1).all()
assert training_data["historical_product_purchase_count"].ge(0).all()
assert training_data["historical_product_unique_customer_count"].ge(0).all()
assert training_data["product_purchase_count_last_30_days"].ge(0).all()
assert (
    training_data["product_purchase_count_last_30_days"]
    <= training_data["historical_product_purchase_count"]
).all()
assert (
    training_data["historical_product_unique_customer_count"]
    <= training_data["historical_product_purchase_count"]
).all()
replenishment_cycle_mask = candidate_data[
    "_has_replenishment_cycle"
]
assert candidate_data.loc[
    ~replenishment_cycle_mask,
    "expected_days_before_next_order",
].eq(0).all()
assert candidate_data.loc[
    ~replenishment_cycle_mask,
    [
        "std_days_between_customer_product_purchases",
        "observed_reorder_interval_count",
    ],
].eq(0).all().all()

group_label_summary = (
    training_data
    .groupby("group_id", sort=False)["label"]
    .agg(group_size="size", positive_count="sum")
)
assert len(group_label_summary) == expected_groups
assert group_label_summary["positive_count"].gt(0).all()
assert (
    group_label_summary["positive_count"]
    < group_label_summary["group_size"]
).all()
assert len(group_metadata) == expected_groups
assert group_metadata["group_id"].is_unique
assert not training_data.duplicated(["group_id", "product_id"]).any()
assert (
    candidate_data["_scoring_date"]
    >= candidate_data["product_id"].map(product_first_seen)
).all()

training_output_path.parent.mkdir(parents=True, exist_ok=True)
training_data.to_csv(training_output_path, index=False)
group_metadata.to_csv(
    group_metadata_path,
    index=False,
    date_format="%Y-%m-%d",
)

saved_training_data = pd.read_csv(
    training_output_path,
    dtype={
        column: dtype
        for column, dtype in identifier_dtypes.items()
        if column in training_columns
    },
)
saved_group_metadata = pd.read_csv(
    group_metadata_path,
    dtype={"customer_id": "string"},
    parse_dates=["scoring_date"],
)
assert list(saved_training_data.columns) == training_columns
assert len(saved_training_data) == len(training_data)
assert len(saved_group_metadata) == len(group_metadata)

first_purchase_positive_rows = int(
    (
        positive_mask
        & ~candidate_data["_had_previous_paid_purchase"]
    ).sum()
)
validation_summary = pd.Series({
    "training_rows": len(training_data),
    "ranking_groups": training_data["group_id"].nunique(),
    "positive_rows": expected_positive_rows,
    "first_purchase_positive_rows": first_purchase_positive_rows,
    "repeat_purchase_positive_rows": (
        expected_positive_rows - first_purchase_positive_rows
    ),
    "historical_negative_rows": int(historical_negative_mask.sum()),
    "never_paid_negative_rows": int(never_paid_negative_mask.sum()),
    "positive_rate": training_data["label"].mean(),
    "mean_candidates_per_group": group_label_summary["group_size"].mean(),
    "median_candidates_per_group": group_label_summary["group_size"].median(),
    "minimum_candidates_per_group": group_label_summary["group_size"].min(),
    "maximum_candidates_per_group": group_label_summary["group_size"].max(),
    "groups_at_target_candidates": group_label_summary["group_size"].eq(
        target_candidates_per_group
    ).mean(),
})

print(f"Saved CatBoost training data to {training_output_path}")
print(f"Saved group metadata to {group_metadata_path}")
validation_summary

Saved CatBoost training data to d:\programming\enterprise-product-recommendation-system\data\processed\catboost_training.csv
Saved group metadata to d:\programming\enterprise-product-recommendation-system\data\processed\catboost_group_metadata.csv


training_rows                    1.062331e+06
ranking_groups                   4.249300e+04
positive_rows                    7.191600e+04
first_purchase_positive_rows     2.085300e+04
repeat_purchase_positive_rows    5.106300e+04
historical_negative_rows         2.614510e+05
never_paid_negative_rows         7.289640e+05
positive_rate                    6.769641e-02
mean_candidates_per_group        2.500014e+01
median_candidates_per_group      2.500000e+01
minimum_candidates_per_group     2.500000e+01
maximum_candidates_per_group     2.900000e+01
groups_at_target_candidates      9.999294e-01
dtype: float64

## Next step

Use the separate group metadata to keep every customer's complete history in one deterministic train, validation, or test split. Pass `label` as the target and `group_id` as CatBoost ranking metadata; neither is an ordinary feature. The remaining 18 saved feature columns are model inputs.